# SmolLM2-360M — DRAG Knowledge Distillation (GKD)

**Student**: `HuggingFaceTB/SmolLM2-360M-Instruct` (~0.7 GB fp16)  
**Teacher**: `meta-llama/Llama-3.1-8B-Instruct` (~16 GB fp16, already approved)  
**Method**: Generalized Knowledge Distillation (GKD) via TRL `GKDTrainer`  
**Target runtime**: H100 96 GB GPU (~1–2 hours)

## Why 8B instead of 70B?
Llama 3.3 70B with 4-bit NF4 (`bitsandbytes`) OOMs on H100 because `bitsandbytes`
materializes the full bf16 weights (~140 GB) in-place before quantizing, consuming all
96 GB VRAM before the student model can even load.

**Llama 3.1 8B is the right teacher here:**
- 22× size ratio (8B → 360M) is within the optimal KD range (studies show 5–30× works best)
- Loads entirely in fp16 (~16 GB), leaving ~78 GB free for student + optimizer + activations
- **True token-level KL divergence is preserved** — this is what makes GKD better than plain SFT
- Llama 3.1 8B is instruction-tuned and follows concise-answer prompts reliably
- Already approved: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct

## VRAM budget
| Component | Size |
|---|---|
| Llama 3.1 8B (fp16) | ~16 GB |
| SmolLM2-360M (bf16 training) | ~0.7 GB |
| Optimizer states (AdamW) | ~1.4 GB |
| Activations + KV cache | ~5 GB |
| **Total** | **~23 GB** (72 GB headroom on H100) |

## What makes this real distillation
The teacher is loaded alongside the student and generates **soft-label logits live during
each training step**. The student minimizes:
- **KD loss** (λ): forward KL divergence between student and teacher token distributions
- **SFT loss** (1-λ): cross-entropy on original curated answers

Vocabulary mismatch (Llama ≠ SmolLM2) is handled at the sequence level:
teacher generates text → decoded to string → re-tokenized by student tokenizer.

## Prerequisites
- Llama 3.1 approved at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct ✅
- HF token with read access in Cell 2
- Google Drive mounted — data at `MyDrive/ColabDrive/Portfolio/`
  - `training_data.jsonl` — 96 original SFT examples
  - `training_data_drag.jsonl` — ~400 DRAG-format examples (run `npm run gen:all-training` locally first)

## Steps
1. Run `npm run gen:all-training` locally to generate triples + DRAG data
2. Upload both JSONL files to Drive at `MyDrive/ColabDrive/Portfolio/`
3. Run Cell 1 (install)
4. Run Cell 2 (HF login)
5. Run Cell 3 (mount Google Drive)
6. Run remaining cells in order
7. Checkpoint saves to `MyDrive/ColabDrive/Portfolio/checkpoint_kd/`
8. ONNX saves to `MyDrive/ColabDrive/Portfolio/onnx-export/`

# Developer Notes

## 1) The "Vocabulary Wall" (Showstopper)

This was the most critical finding. We initially tried `GKDTrainer` (Generalized Knowledge Distillation), which typically compares teacher and student probability distributions (logits).

- **Error:** `RuntimeError: stack expects each tensor to be equal size`
- **Lesson:** Standard online logit-based distillation does not work between Llama 3 (128k vocab) and SmolLM2 (49k vocab). Their output vectors have different lengths and cannot be compared 1:1.
- **Fix:** Switched to **Offline Distillation** (Cell 7). The teacher generates answers, and the student trains on those outputs. This works regardless of tokenizer mismatch.

## 2) Library "Bleeding Edge" Pains

We needed a very new TRL feature (`GKDTrainer`).

- **Missing imports:** Even with `pip install trl>=0.12.0`, the class was unavailable; installed from source (`git+github...`).
- **Hidden path:** Class was not in the main module; found under `trl.experimental.gkd`.
- **Unstable APIs:** Arguments changed; `max_seq_length` was rejected by both config and trainer, so defaults were used.

## 3) Dependency Hell (Training vs. Export)

- **Conflict:** Training stack (latest `transformers`, `trl`) conflicted with export stack (`optimum-onnx` preferring older `transformers`).
- **Solution:** Separated phases. Keep training dependencies in Cell 1; install export tooling in Cell 10 after training.

In [ ]:
# Cell 1 — Install dependencies
# Uninstall conflicting packages first to ensure a clean slate
!pip uninstall -y optimum-onnx optimum trl

# GKDTrainer is in trl>=0.12.0. We install from source to be safe.
!pip install -q git+https://github.com/huggingface/trl.git

# Install main libraries
!pip install -q -U transformers datasets accelerate bitsandbytes huggingface_hub

# For ONNX export, install optimum and onnxruntime.
# We avoid 'optimum[onnx]' package which has strict pinning and use main 'optimum' instead.
!pip install -q optimum onnx onnxruntime

print('Installation complete.')

In [ ]:
# Cell 2 — HF login (for uploading checkpoint to Hub)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Cell 3 — Mount Google Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT   = '/content/drive/MyDrive/ColabDrive/Portfolio'
# Both datasets: original SFT Q&A + new DRAG-format (system+evidence+triples)
DATA_PATHS   = [
    f'{DRIVE_ROOT}/training_data.jsonl',       # 96 original SFT examples
    f'{DRIVE_ROOT}/training_data_drag.jsonl',  # ~400 DRAG-format examples
]
CKPT_DIR     = f'{DRIVE_ROOT}/checkpoint_kd'    # separate from SFT checkpoint
ONNX_DIR     = f'{DRIVE_ROOT}/onnx-export'

import os
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(ONNX_DIR, exist_ok=True)

print(f'Checkpoint: {CKPT_DIR}')
print(f'ONNX:       {ONNX_DIR}')
for p in DATA_PATHS:
    print(f'  {"✅" if os.path.exists(p) else "❌"} {p}')

In [ ]:
# Cell 4 — Load and combine both datasets
# training_data.jsonl       — 96 original SFT pairs (user/assistant only)
# training_data_drag.jsonl  — ~400 DRAG-format pairs (system+evidence+triples → answer)
# Combining both gives the student: direct Q&A recall + grounded evidence reasoning.
import json
from datasets import Dataset

rows = []
for path in DATA_PATHS:
    if not os.path.exists(path):
        print(f'⚠️  Skipping missing file: {path}')
        continue
    before = len(rows)
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            ex = json.loads(line)
            msgs = ex['messages']
            rows.append({'messages': msgs})
    print(f'Loaded {len(rows) - before} examples from {os.path.basename(path)}')

print(f'\nTotal examples: {len(rows)}')
dataset = Dataset.from_list(rows)

# 90/10 train/eval split
split = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split['train']
eval_ds  = split['test']

print(f'Train: {len(train_ds)} | Eval: {len(eval_ds)}')

In [ ]:
# Cell 5 — Load teacher (Llama 3.1 8B) and student (SmolLM2-360M)
#
# Teacher in fp16 uses ~16 GB — leaves 78+ GB free for student + optimizer.
# No quantization needed; fp16 gives full logit fidelity for KD.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

TEACHER_ID = 'meta-llama/Llama-3.1-8B-Instruct'
STUDENT_ID = 'HuggingFaceTB/SmolLM2-360M-Instruct'

print('Loading teacher tokenizer...')
teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_ID)

print('Loading teacher model (Llama 3.1 8B, fp16)...')
teacher_model = AutoModelForCausalLM.from_pretrained(
    TEACHER_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)
teacher_model.eval()

allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved()  / 1e9
print(f'Teacher loaded. VRAM allocated: {allocated:.1f} GB | reserved: {reserved:.1f} GB')

print('Loading student model (bf16 for training stability + H100 efficiency)...')
student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_ID)
if student_tokenizer.pad_token is None:
    student_tokenizer.pad_token = student_tokenizer.eos_token

student_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
student_model.train()

allocated2 = torch.cuda.memory_allocated() / 1e9
free_gb = (torch.cuda.get_device_properties(0).total_memory / 1e9) - allocated2
print(f'Both loaded. VRAM used: {allocated2:.1f} GB | free: {free_gb:.1f} GB')
print(f'Teacher params: {sum(p.numel() for p in teacher_model.parameters()) / 1e9:.1f}B')
print(f'Student params: {sum(p.numel() for p in student_model.parameters()) / 1e6:.0f}M')

In [ ]:
# Cell 6 — Format dataset as chat-templated text (for GKDTrainer prompt/completion split)
# ⚠️  SKIP THIS CELL — only needed for the old GKD pipeline.
# Cell 7b (direct SFT) expects raw 'messages' dicts. Running this cell adds
# prompt/completion string columns that confuse SFTTrainer.
# GKDTrainer expects 'prompt' (user turn only) so the teacher can generate the completion.

def format_prompt(example):
    msgs = example['messages']
    # Prompt = everything except the last assistant turn
    prompt_msgs = [m for m in msgs if m['role'] != 'assistant']
    prompt = student_tokenizer.apply_chat_template(
        prompt_msgs,
        tokenize=False,
        add_generation_prompt=True,
    )
    # Teacher-formatted prompt — Llama uses its own chat template, not ChatML
    teacher_prompt = teacher_tokenizer.apply_chat_template(
        prompt_msgs,
        tokenize=False,
        add_generation_prompt=True,
    )
    # Completion = the original assistant answer (used as SFT reference)
    completion = next((m['content'] for m in reversed(msgs) if m['role'] == 'assistant'), '')
    return {'prompt': prompt, 'teacher_prompt': teacher_prompt, 'completion': completion}

train_ds = train_ds.map(format_prompt)
eval_ds  = eval_ds.map(format_prompt)

print('Sample prompt:')
print(train_ds[0]['prompt'])
print('Sample completion:')
print(train_ds[0]['completion'])

In [ ]:
# Cell 7a — Offload teacher (not needed for direct SFT)
# ──────────────────────────────────────────────────────────────────────────────
# LESSON LEARNED: Offline distillation (teacher generates → student trains on
# teacher text) does NOT add value when:
#   1. Vocab mismatch blocks online KD (no soft-label transfer possible)
#   2. The curated answers (from 70B via Groq) are already higher quality
#      than what the 8B teacher would generate
#   3. Offline KD only gives hard labels — identical to plain SFT
#
# We now train directly on the curated data from both JSONL files.

import gc
import torch
from trl import SFTTrainer, SFTConfig

print('Offloading teacher to free VRAM for SFT...')
teacher_model.cpu()
del teacher_model, teacher_tokenizer
gc.collect()
torch.cuda.empty_cache()

free_gb = (
    torch.cuda.get_device_properties(0).total_memory
    - torch.cuda.memory_allocated()
) / 1e9
print(f'VRAM after teacher offload: {free_gb:.1f} GB free')


In [ ]:
# Cell 7b — Direct SFT on curated training data
# ──────────────────────────────────────────────────────────────────────────────
# Train the student directly on the curated Q&A pairs from both JSONL files.
# The answers were generated by Llama 3.3 70B (via Groq) and verified with
# grounding scores ≥ 0.85 — no need for an 8B teacher to re-generate them.
#
# Key fixes vs. previous version:
#   - Train on curated data (train_ds), not teacher-generated synthetic_ds
#   - 8 epochs with load_best_model_at_end (best checkpoint auto-selected around epoch 6)
#   - Eval every epoch with load_best_model_at_end to catch overfitting
#   - learning_rate 2e-5 (standard SFT) instead of 3e-6 (too conservative)
#   - completion_only_loss=True masks prompt tokens (TRL handles template detection)
#   - max_grad_norm=1.0 (default) instead of 0.3 (too restrictive)

print('\n--- Direct SFT on curated data ---')

# Safety: if train_ds got corrupted by a previous run (e.g. Cell 6 added
# string columns, or _format_to_text removed messages), reload from disk.
if 'messages' not in train_ds.column_names or len(train_ds) == 0:
    print('⚠️  train_ds missing messages column or empty — reloading from disk...')
    import json as _json
    from datasets import Dataset as _Dataset
    _rows = []
    for _p in DATA_PATHS:
        if not os.path.exists(_p): continue
        with open(_p) as _f:
            for _line in _f:
                _line = _line.strip()
                if _line:
                    _rows.append({'messages': _json.loads(_line)['messages']})
    _split = _Dataset.from_list(_rows).train_test_split(test_size=0.1, seed=42)
    train_ds, eval_ds = _split['train'], _split['test']
    print(f'  Reloaded {len(train_ds)} train / {len(eval_ds)} eval examples')

# Strip extra columns from Cell 6 if present
drop = [c for c in train_ds.column_names if c != 'messages']
if drop:
    print(f'Dropping extra columns: {drop}')
    train_ds = train_ds.remove_columns(drop)
    eval_ds  = eval_ds.remove_columns(drop)

# Fix: HF Datasets stores messages as Arrow structs. TRL's Jinja2 template
# can fail with 'str has no attribute role' on Arrow structs. Pre-format to
# text ourselves and use a manual completion-only collator.
def _format_to_text(example):
    msgs = [dict(m) for m in example['messages']]
    return {'text': student_tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=False
    )}

train_ds = _Dataset.from_list([_format_to_text(ex) for ex in train_ds])
eval_ds  = _Dataset.from_list([_format_to_text(ex) for ex in eval_ds])

print(f'Train: {len(train_ds)} examples | Eval: {len(eval_ds)} examples')
assert len(train_ds) > 0, 'No training data! Re-run Cell 5 or check DATA_PATHS.'
print(f'Sample (first 200 chars): {train_ds[0]["text"][:200]}')

# ── Completion-only data collator ─────────────────────────────────────────
# TRL's completion_only_loss=True is broken in the current version (loss ~2.5
# instead of ~0.95). Implement manually: mask all tokens before the LAST
# '<|im_start|>assistant\n' boundary so the model only trains on completions.
from transformers import DataCollatorForLanguageModeling
import torch

RESPONSE_TEMPLATE = student_tokenizer.encode(
    '<|im_start|>assistant\n', add_special_tokens=False
)
print(f'Response template token IDs: {RESPONSE_TEMPLATE}')

class CompletionOnlyCollator(DataCollatorForLanguageModeling):
    def torch_call(self, examples):
        batch = super().torch_call(examples)
        resp = RESPONSE_TEMPLATE
        for i in range(len(batch['labels'])):
            seq = batch['labels'][i].tolist()
            # Find LAST occurrence of the response template
            idx = -1
            for j in range(len(seq) - len(resp) + 1):
                if seq[j:j+len(resp)] == resp:
                    idx = j + len(resp)
            if idx > 0:
                batch['labels'][i, :idx] = -100
        return batch

completion_collator = CompletionOnlyCollator(
    tokenizer=student_tokenizer, mlm=False
)

# ── SFT config ───────────────────────────────────────────────────────────
sft_config = SFTConfig(
    output_dir=CKPT_DIR,
    num_train_epochs=8,               # 8 epochs; load_best_model_at_end picks optimal checkpoint
    per_device_train_batch_size=16,    # ~80 GB free — 16 is safe with 2048 seq len
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,     # effective batch = 32
    learning_rate=2e-5,                # standard SFT rate (was 3e-6 — too slow)
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    max_grad_norm=1.0,                 # default (was 0.3 — too restrictive)
    bf16=True,
    logging_steps=5,
    report_to='none',
    save_strategy='epoch',
    eval_strategy='epoch',             # track eval loss every epoch
    load_best_model_at_end=True,        # pick best checkpoint by eval loss
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    max_seq_length=2048,
    dataset_text_field='text',          # pre-formatted text
)

trainer = SFTTrainer(
    model=student_model,
    args=sft_config,
    train_dataset=train_ds,            # curated data (was synthetic_ds)
    eval_dataset=eval_ds,              # eval split (was missing)
    data_collator=completion_collator, # manual completion-only loss masking
    processing_class=student_tokenizer,
)

print('\nStarting SFT (completion-only loss — prompt tokens masked)...')
trainer.train()
print('SFT complete.')

# Print final train/eval loss
metrics = trainer.state.log_history
train_losses = [m['loss'] for m in metrics if 'loss' in m]
eval_losses  = [m['eval_loss'] for m in metrics if 'eval_loss' in m]
if train_losses:
    print(f'Final train loss: {train_losses[-1]:.4f}')
if eval_losses:
    print(f'Best eval loss:   {min(eval_losses):.4f}')
    print(f'Eval loss trend:  {" → ".join(f"{l:.4f}" for l in eval_losses)}')


In [ ]:
# Cell 8 — Quick qualitative eval
import torch
from transformers import GenerationConfig

model = trainer.model
model.eval()

# Reset generation config to avoid any max_length=20 artifact saved during SFT
model.generation_config = GenerationConfig(
    max_new_tokens=200,
    do_sample=False,
    pad_token_id=student_tokenizer.eos_token_id,
    eos_token_id=student_tokenizer.eos_token_id,
)

system = 'You are a helpful assistant for Kham\'s portfolio. Be concise and accurate.'

prompts = [
    'What are Kham\'s main technical skills?',
    'How many years of experience does Kham have in AI/ML?',
    'What companies has Kham worked at?',
    'Tell me about Kham\'s DRAG project.',
]

print("=== Fine-tuned SmolLM2-360M Qualitative Eval ===\n")
for q in prompts:
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': q}]
    text = student_tokenizer.apply_chat_template(
        msgs,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = student_tokenizer(text, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=200, do_sample=False,
                                  pad_token_id=student_tokenizer.eos_token_id)

    # Decode only the newly generated tokens
    new_tokens = out_ids[0][input_len:]
    answer = student_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    print(f"Q: {q}")
    print(f"A: {answer if answer else '[EMPTY — model generated EOS immediately]'}")
    print()

In [ ]:
# Cell 8b — Eval WITH evidence (how the model is actually used in production)
system_with_evidence = """You are an AI assistant for Kham's professional portfolio.

## Evidence
1. Kham is a Data Scientist at Florida Power & Light (NextEra Energy). He has 14 years of semiconductor hardware expertise and 5+ years deploying
production AI/ML systems. His skills include RF chip design, thin-film fabrication, EM simulation, multi-agent systems, RAG, LLM fine-tuning.
2. Kham worked at Smiths Interconnect (2005-2011), Nova Microwave (2011-2017), and Florida Power & Light (2022-present). He holds degrees from
University of South Florida.

Answer questions about Kham based strictly on the evidence above."""

for q in prompts:
    msgs = [{'role': 'system', 'content': system_with_evidence},
            {'role': 'user', 'content': q}]
    text = student_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = student_tokenizer(text, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=200, do_sample=False,
                                    repetition_penalty=1.2,
                                    pad_token_id=student_tokenizer.eos_token_id)
    answer = student_tokenizer.decode(out_ids[0][input_len:], skip_special_tokens=True).strip()
    print(f"Q: {q}\nA: {answer}\n")

In [ ]:
# Cell 9 — Save checkpoint to Drive and push to HF Hub
import os

trainer.save_model(CKPT_DIR)
student_tokenizer.save_pretrained(CKPT_DIR)
print(f'Checkpoint saved to Drive: {CKPT_DIR}')

# Push to HF Hub
HF_REPO = '<your-hf-username>/smollm2-drag'
trainer.model.push_to_hub(HF_REPO, commit_message='Direct SFT on curated data (8 epochs, completion-only loss)')
student_tokenizer.push_to_hub(HF_REPO)
print(f'Pushed to: https://huggingface.co/{HF_REPO}')

In [ ]:
# Cell 10 — Export to ONNX (fp32)
# WORKING COMMAND (confirmed): --dtype fp32 --opset 18
# NOTE: --int8 flag is NOT supported by optimum-cli in this environment.
# NOTE: quantize_dynamic (Cell 11) produces float16 scale tensors that crash ONNX Runtime Web/WASM.
# RECOMMENDATION: Use fp32 export directly. transformers.js dtype: fp32 loads model.onnx correctly.

!pip install -q "optimum[onnx]" onnx onnxruntime

import os
os.makedirs(ONNX_DIR, exist_ok=True)
print(f"Exporting model from {CKPT_DIR} to {ONNX_DIR}...")

!optimum-cli export onnx \
    --model "{CKPT_DIR}" \
    --task text-generation-with-past \
    --device cuda \
    --dtype fp32 \
    --opset 18 \
    "{ONNX_DIR}"

print("ONNX export complete. Files:", os.listdir(ONNX_DIR))

In [ ]:
# (This cell was a scratch replacement — superseded by Cell 10 above.)
# No action needed here.

In [ ]:
# Cell 11 — Quantize ONNX (INT8 / q8)
# ──────────────────────────────────────────────────────────────────────────────
# Produces model_quantized.onnx (~364 MB) from model.onnx (~1.45 GB).
# quantize_dynamic still emits tensor(float16) scale tensors in some nodes.
# Run Cell 11b immediately after to fix them before uploading.

import os
from onnxruntime.quantization import quantize_dynamic, QuantType

for root, dirs, files in os.walk(ONNX_DIR):
    for f in files:
        if f.endswith('.onnx') and 'quantized' not in f:
            src = os.path.join(root, f)
            dst = src.replace('.onnx', '_quantized.onnx')
            print(f'Quantizing: {f}...')
            try:
                quantize_dynamic(
                    model_input=src,
                    model_output=dst,
                    per_channel=False,
                    reduce_range=False,
                    weight_type=QuantType.QInt8
                )
                print(f' -> Success: {os.path.basename(dst)}')
            except Exception as e:
                print(f' -> Failed: {e}')

print('\nDone. Run Cell 11b next to fix float16 scale tensors.')


In [ ]:
# Cell 11b — Fix float16 scale tensors in quantized ONNX for ONNX Runtime Web
# ───────────────────────────────────────────────────────────────────────────
# quantize_dynamic produces tensor(float16) scale tensors in DequantizeLinear
# nodes (especially the embedding layer). ONNX Runtime Web (WASM/WebGPU)
# rejects these: 'Type tensor(float16) of input parameter is invalid.'
#
# Fix: cast all float16 initializers → float32 in-place.
# INT8 weights are unaffected. File size +~1 MB (negligible).
# After this, dtype: 'q8' in transformers.js loads model_quantized.onnx correctly.

import onnx
from onnx import numpy_helper, TensorProto
import numpy as np

def fix_float16_scales(model_path: str) -> int:
    """Convert tensor(float16) initializers → tensor(float32) in-place."""
    model = onnx.load(model_path)
    fixed = 0
    for init in model.graph.initializer:
        if init.data_type == TensorProto.FLOAT16:
            arr = numpy_helper.to_array(init).astype(np.float32)
            new_init = numpy_helper.from_array(arr, name=init.name)
            init.CopyFrom(new_init)
            fixed += 1
    if fixed:
        onnx.save(model, model_path)
    return fixed

print('Fixing float16 scale tensors in quantized ONNX files...')
for root, dirs, files in os.walk(ONNX_DIR):
    for f in files:
        if f.endswith('_quantized.onnx'):
            path = os.path.join(root, f)
            n = fix_float16_scales(path)
            size_mb = os.path.getsize(path) / 1e6
            print(f'  {f}: fixed {n} float16→float32 tensors  ({size_mb:.0f} MB)')

print('\nAll quantized models are now ONNX Runtime Web compatible.')
print("Run Cell 11c to upload to HF Hub.")


In [ ]:
# Cell 11c — Upload fixed model_quantized.onnx to HF Hub
# ─────────────────────────────────────────────────────────────────
# Run Cell 11b first. Browser dtype: 'q8' → loads model_quantized.onnx at repo root.
!pip install -q --force-reinstall huggingface_hub
import os
from huggingface_hub import HfApi

api = HfApi()
HF_REPO = '<your-hf-username>/smollm2-drag'

quantized_path = None
for root, dirs, files in os.walk(ONNX_DIR):
    for f in files:
        if f == 'model_quantized.onnx':
            quantized_path = os.path.join(root, f)
            break
    if quantized_path:
        break

if not quantized_path:
    raise FileNotFoundError('model_quantized.onnx not found. Run Cells 11 + 11b first.')

size_mb = os.path.getsize(quantized_path) / 1e6
print(f'Uploading {quantized_path} ({size_mb:.0f} MB) to {HF_REPO}...')
api.upload_file(
    path_or_fileobj=quantized_path,
    path_in_repo='model_quantized.onnx',
    repo_id=HF_REPO,
    repo_type='model',
    commit_message='Fix float16 scale tensors to float32 for ONNX Runtime Web (WASM) compatibility',
)
print(f'Done! https://huggingface.co/{HF_REPO}')


In [ ]:
# Cell 12 — Upload ONNX to HF Hub
# Uploads the entire ONNX_DIR (model.onnx + model_with_past.onnx + config files).
# transformers.js loads model.onnx with dtype: "fp32" — no quantized file needed.

import os
from huggingface_hub import HfApi

api = HfApi()
HF_REPO = "<your-hf-username>/smollm2-drag"

print(f"Uploading folder {ONNX_DIR} to {HF_REPO}...")
api.upload_folder(
    folder_path=ONNX_DIR,
    repo_id=HF_REPO,
    repo_type="model",
    commit_message="Add fp32 ONNX export (opset 18, direct SFT on curated data)",
)
print(f"Done! https://huggingface.co/{HF_REPO}/tree/main")